# ==========================================================
# Ethiopia Exchange Rate & Inflation Analysis 
# ==========================================================

In [5]:
# installing important packages
# !pip install requests
# !pip install pandas
# !pip install matplotlib

In [1]:
# ----------------------------
#  Import Libraries
# ----------------------------
import pandas as pd
import matplotlib.pyplot as plt
import requests

### 💱 Fetching ETB/USD Exchange Rates

This script retrieves historical annual exchange rate data for the **Ethiopian Birr (ETB)** against the **US Dollar (USD)** using the [World Bank Open Data API](https://data.worldbank.org/).

#### API Configuration
- **Indicator**: `PA.NUS.FCRF` (Official exchange rate, LCU per US$, period average).
- **Endpoint**: Ethiopia (`ETH`) specific time-series.
- **Format**: JSON (returning the 100 most recent records).

#### Data Access
The response is returned as a nested list:
* `data_rate[0]`: Contains API metadata and pagination.
* `data_rate[1]`: Contains the actual observation list (Year and Rate).

In [ ]:
countries = {
    "Ethiopia": "ETH",
    "Kenya": "KEN",
    "Japan": "JPN",
    "Brazil": "BRA",
    "India": "IND"
}

exchange_indicator = "PA.NUS.FCRF"
inflation_indicator = "FP.CPI.TOTL.ZG"

all_data = []

for country, code in countries.items():
    # Exchange rates
    url_ex = f"https://api.worldbank.org/v2/country/{code}/indicator/{exchange_indicator}?format=json&per_page=1000"
    ex_data = requests.get(url_ex).json()[1]
    
    # Inflation
    url_inf = f"https://api.worldbank.org/v2/country/{code}/indicator/{inflation_indicator}?format=json&per_page=1000"
    inf_data = requests.get(url_inf).json()[1]
    
    # Merge by year
    for ex, inf in zip(ex_data[::-1], inf_data[::-1]):  # reverse to chronological
        if ex["value"] is not None and inf["value"] is not None:
            all_data.append({
                "Country": country,
                "Year": int(ex["date"]),
                "Exchange Rate": ex["value"],
                "Inflation (%)": inf["value"]
            })

df = pd.DataFrame(all_data)
df = df.sort_values(["Country", "Year"])
df.to_csv("exchange_and_inflation.csv", index=False)

print(df.head())

In [ ]:
plt.figure(figsize=(12,6))

for country in df["Country"].unique():
    subset = df[df["Country"] == country]
    
    plt.plot(subset["Year"], subset["Exchange Rate"], label=country)


plt.legend()
plt.title("Exchange Rate Comparison")
plt.xlabel("Year")
plt.ylabel("Local Currency per USD")
plt.grid(True)
plt.xticks(rotation=90)  # rotate labels so they are readable
plt.show()

In [ ]:
for country in df["Country"].unique():
    subset = df[df["Country"] == country]
    
    fig, ax1 = plt.subplots(figsize=(14,6))
    
    color = "tab:blue"
    ax1.set_xlabel("Year")
    ax1.set_ylabel("Exchange Rate", color=color)
    ax1.plot(subset["Year"], subset["Exchange Rate"], color=color, label="Exchange Rate")
    ax1.tick_params(axis='y', labelcolor=color)
    
    ax2 = ax1.twinx()  # second y-axis for inflation
    color = "tab:red"
    ax2.set_ylabel("Inflation (%)", color=color)
    ax2.plot(subset["Year"], subset["Inflation (%)"], color=color, label="Inflation")
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title(f"{country}: Exchange Rate vs Inflation")
    fig.tight_layout()
    plt.show()